### Install dependencies

In [108]:
!pip install chromadb
!pip install torch
!pip install sentence-transformers

### Import dependencies

In [109]:
import numpy
import pandas as pd
import json
import re
import requests
import torch

### Define Helper functions

In [110]:
def get_anchored_examples_text(reference_pool_df, samples_per_group=2):
    """Dynamically extracts historical baseline data records from the reference pool dataframe

    by auto-discovering numerical feature tracking schema columns, enforces token brevity via
    precision rounding, and formats the data with standard tracking header tokens.
    """
    try:
        dynamic_features = [
            col
            for col in reference_pool_df.columns
            if pd.api.types.is_numeric_dtype(reference_pool_df[col])
        ]

        if not dynamic_features:
            print(
                "[Archival Warning] No numeric features detected. Falling back"
                " to all schema features."
            )
            dynamic_features = list(reference_pool_df.columns)

        # Extract a clean isolated slice using the dynamically detected columns
        isolated_pool = reference_pool_df[dynamic_features].copy()

        # Pull a representative random selection across our pipeline pool rows
        sample_size = min(int(samples_per_group), len(isolated_pool))
        sampled_records = isolated_pool.sample(
            n=sample_size, random_state=42
        ).reset_index(drop=True)

        # Enforce precise 2-decimal constraints across all selected numeric series data spaces
        for col in sampled_records.columns:
            if pd.api.types.is_numeric_dtype(sampled_records[col]):
                sampled_records[col] = sampled_records[col].round(2)

        # 5. Serialize records array into a raw JSON array string
        records_dict_list = sampled_records.to_dict(orient="records")
        json_array_string = json.dumps(records_dict_list, indent=2)

        # 6. Structure output layout with the explicit marker header token used by parsing engines
        formatted_context_block = (
            "=== SYSTEM TELEMETRY ARCHIVAL MANIFOLD REFERENCE ===\n"
            "This reference data block defines the valid physical vocabulary"
            " boundaries.\n\n"
            "REAL HISTORICAL EXAMPLES FROM DATASETS (ROUNDED FOR CONTEXT):\n"
            f"{json_array_string}"
        )

        return formatted_context_block

    except Exception as caught_error:
        print(
            "[Archival Context Extraction Error] Failed to dynamically"
            f" compile structural context block: {caught_error}"
        )
        # Resilient fallback state matching structural reflection token patterns
        return (
            "REAL HISTORICAL EXAMPLES FROM DATASETS (ROUNDED FOR CONTEXT):\n"
            "[\n  {\n    \"Variable_1\": 0.0,\n    \"Variable_2\": 0.0\n  }\n]"
        )

### Define LLM endpoint and Model ID (for Ollama use)

In [ ]:
GENERATE_ENDPOINT = "https://swimming-adjusted-ebony-collaboration.trycloudflare.com/api/generate"
MODEL_ID = "qwen3.5:9b"

In [ ]:
df = pd.read_csv("golf_data.csv")
df.columns

Index(['Club', 'Ball', 'Club Speed', 'Attack Angle', 'Club Path', 'Low Point',
       'Swing Plane', 'Swing Direction', 'Dyn. Loft', 'Face Angle',
       'Face To Path', 'Ball Speed', 'Smash Factor', 'Launch Angle',
       'Launch Direction', 'Spin Rate', 'Spin Axis', 'Max Height - Dist.',
       'Max Height - Height', 'Max Height - Side', 'Last data Point - Length',
       'Last data Point - Side', 'Last data Point - Height',
       'Last data Point - Time', 'Carry Flat - Length', 'Carry Flat - Side',
       'Carry Flat - Land. Angle', 'Carry Flat - Ball Speed',
       'Carry Flat - Time', 'Total', 'Side Total', 'Dynamic Lie',
       'Impact Offset', 'Impact Height', 'Curve'],
      dtype='object')

### Define challenger agent

In [112]:
CHALLENGER_PROMPT_TEMPLATE = """
Role: You are an expert data scientist and tracking matrix challenge designer. You are provided with a reference block containing valid categorical vocabularies and real historical examples extracted from a tracking dataset. 

Task: Brainstorm and output ONE highly specific, difficult, or extreme edge-case scenario described in natural language text, alongside its matching explicit target numerical values.

Instructions:
1. Read the available allowed vocabulary and data rows provided below.
2. Write a detailed descriptive prompt targeting specific combinations of independent tracking variables to define a unique statistical challenge.
3. CRITICAL: You must strictly anchor your scenario to the exact columns and valid values provided in the reference text below. Do NOT reference features, contexts, or environments that do not exist in this tracking schema.

Output Schema Layout:
{{
    "scenario": "A detailed descriptive natural language prompt targeting specific delivery variables...",
    "target_numerical_dict": {{
{expected_keys_outline}
    }}
}}

REFERENCE DATA (VALID VOCABULARY & EXAMPLES):
{anchored_examples}
"""

class Challenger:
    def __init__(self, llm_url, model_id, anchored_examples="", prompt_template=None):
        """
        A domain-agnostic synthetic data challenge agent that auto-discovers
        and maps all schema variables from the provided data text context.
        """
        self.api_url = llm_url
        self.model_id = model_id
        self.historical_sample_rows = anchored_examples
        self.prompt_template = prompt_template if prompt_template is not None else CHALLENGER_PROMPT_TEMPLATE

    def _extract_all_columns(self):
        try:
            marker = "REAL HISTORICAL EXAMPLES FROM DATASETS (ROUNDED FOR CONTEXT):"
            if marker in self.historical_sample_rows:
                json_part = self.historical_sample_rows.split(marker)[1].strip()
                sample_records = json.loads(json_part)
                if sample_records and isinstance(sample_records, list):
                    return list(sample_records[0].keys())
        except Exception:
            pass
        
        return ["Variable_1", "Variable_2"]

    def _get_keys_outline(self):
        columns = self._extract_all_columns()
        lines = []
        for i, col in enumerate(columns):
            comma = "," if i < len(columns) - 1 else ""
            if col in ['Club', 'Ball']:
                lines.append(f'        "{col}": "string value"{comma}')
            else:
                lines.append(f'        "{col}": 0.0{comma}')  # Changed to 0.0 to help Qwen type-matching
        return "\n".join(lines)

    def _build_prompt(self, previous_failures=None, failure_mode=None):
        keys_outline = self._get_keys_outline()
        prompt = self.prompt_template.format(
            expected_keys_outline=keys_outline,
            anchored_examples=self.historical_sample_rows
        )

        if failure_mode:
            prompt += f"\nMODE: IMPROVE\nFeedback from previous attempt to correct: {failure_mode}\n"

        if previous_failures:
            prompt += "\nPrevious failed text scenarios to absolutely avoid:\n"
            for fail in previous_failures:
                prompt += f"- {fail}\n"

        return prompt

    def generate(self, previous_failures=None, failure_mode=None):
        print("[data_challenger] Compiling contextual prompt...", flush=True)
        prompt = self._build_prompt(previous_failures, failure_mode)
        
        payload = {
            "model": self.model_id,
            "prompt": prompt,
            "stream": False,
            "format": "json",
            "options": {
                "temperature": 0.5, 
                "num_ctx": 16384
            }
        }
        
        print(f"[data_challenger] Dispatching task to Ollama ({self.model_id})...", flush=True)
        raw_output = ""
        try:
            response = requests.post(self.api_url, json=payload, timeout=300)
            response.raise_for_status() 
            
            result_json = response.json()
            
            raw_output = result_json.get("response", "").strip()
            if not raw_output and "thinking" in result_json:
                raw_output = result_json.get("thinking", "").strip()
            
            cleaned_text = re.sub(r'^```[a-zA-Z]*\s*|\s*```$', '', raw_output).strip()
            
            if not (cleaned_text.startswith('{') and cleaned_text.endswith('}')):
                match = re.search(r'(\{.*\})', cleaned_text, re.DOTALL)
                if match:
                    cleaned_text = match.group(1).strip()
            
            data = json.loads(cleaned_text)
            scenario = data.get("scenario", "").strip()
            target_numerical_dict = data.get("target_numerical_dict", {})
            
            # Catch empty template echoes explicitly
            if not scenario or target_numerical_dict == {}:
                print(f"[data_challenger] Warning: Model returned an unpopulated schema template map.")
                return None
                
            return scenario, target_numerical_dict
            
        except (requests.exceptions.RequestException, json.JSONDecodeError, KeyError) as e:
            print(f"[data_challenger] Generation or parsing error occurred: {e}", flush=True)
            print(f"Raw model response context was:\n{raw_output}\n---", flush=True)
            return None

### Define Retriever with min-max optimization
Use CUDA for embeddings if avaliable

In [113]:
import chromadb
from chromadb.utils import embedding_functions

# Safe conditional CUDA hardware selection
try:
    compute_device = "cuda" if torch.cuda.is_available() else "cpu"
except ImportError:
    compute_device = "cpu"

# Initialize local embedding calculation space
EMBEDDING_FUNC = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2", device=compute_device
)

class Retriever:

    def __init__(self, historical_data, feature_columns=None):
        """Initializes a domain-agnostic vector storage space, indexes tracking telemetry,

        and manages high-dimensional validation lookups across all columns.
        """
        self.chroma_client = chromadb.Client()
        self.vector_collection = self.chroma_client.create_collection(
            name="historical_data_registry",
            embedding_function=EMBEDDING_FUNC,
            get_or_create=True,
        )

        # Explicit feature selection or automatic preservation of all columns
        if feature_columns is not None:
            self.tracking_features = feature_columns
        else:
            self.tracking_features = list(historical_data.columns)

        self._populate_vector_database(historical_data)

    def _serialize_row_to_text(self, data_dict):
        """Converts an index tracking record cleanly into descriptive text strings."""
        features_summary = ", ".join(
            [f"{key}: {val}" for key, val in data_dict.items() if val != ""]
        )
        return f"Tracking Data Profile -> {features_summary}."

    def _populate_vector_database(self, historical_data):
        """Batch-serializes records and maps them into local vector collections."""
        print(
            f"[Retriever] Preparing vector indexing for {len(historical_data)} data rows...",
            flush=True,
        )

        # Vectorized replacement of missing values for consistent downstream typing
        cleaned_dataframe = historical_data.fillna("")

        text_documents = []
        metadata_records = []
        entry_ids = []

        for record_index, raw_row in cleaned_dataframe.iterrows():
            row_dict = raw_row.to_dict()

            # Attain numeric structure representations where cleanly parseable
            for feature in self.tracking_features:
                if feature in row_dict and row_dict[feature] != "":
                    try:
                        row_dict[feature] = float(row_dict[feature])
                    except (ValueError, TypeError):
                        pass  # Retain raw formatting state for categorical rows

            text_documents.append(self._serialize_row_to_text(row_dict))
            metadata_records.append(row_dict)
            entry_ids.append(f"record_{record_index}")

        # Chunk indexing runs to optimize CPU/RAM efficiency thresholds
        total_records = len(text_documents)
        batch_slice_size = 1000

        for batch_start in range(0, total_records, batch_slice_size):
            batch_end = min(batch_start + batch_slice_size, total_records)

            self.vector_collection.add(
                documents=text_documents[batch_start:batch_end],
                metadatas=metadata_records[batch_start:batch_end],
                ids=entry_ids[batch_start:batch_end],
            )
            print(
                f"[Retriever Progress] Indexed {batch_end}/{total_records} tracking rows.",
                flush=True,
            )

    def retrieve(
        self,
        scenario_description,
        target_metrics_dict=None,
        semantic_pool_size=30,
        top_matches_count=3,
    ):
        """Executes a two-stage hybrid retrieval pass:

        1. Coarse semantic text embedding evaluation over natural descriptions.
        2. Localized distance evaluation supporting cross-type metrics (numeric and text).
        """
        # Defensive check for missing dictionary arguments
        if target_metrics_dict is None:
            target_metrics_dict = {}

        print(
            f"[Retriever] Evaluating primary semantic clusters (Pool Size: {semantic_pool_size})...",
            flush=True,
        )

        semantic_search_results = self.vector_collection.query(
            query_texts=[scenario_description], n_results=semantic_pool_size
        )

        if (
            not semantic_search_results
            or not semantic_search_results.get("metadatas")
            or len(semantic_search_results["metadatas"][0]) == 0
        ):
            return []

        retrieved_candidates = semantic_search_results["metadatas"][0]

        # Calculate localized scaling variance parameters strictly for valid numeric targets
        feature_variance_bounds = {}
        for feature in self.tracking_features:
            try:
                valid_pool_values = [
                    float(candidate.get(feature))
                    for candidate in retrieved_candidates
                    if candidate.get(feature) is not None
                    and candidate.get(feature) != ""
                ]
                if valid_pool_values:
                    min_val, max_val = min(valid_pool_values), max(
                        valid_pool_values
                    )
                    span_range = (
                        (max_val - min_val) if (max_val - min_val) > 0 else 1.0
                    )
                    feature_variance_bounds[feature] = {
                        "minimum": min_val,
                        "range": span_range,
                    }
                else:
                    feature_variance_bounds[feature] = {
                        "minimum": 0.0,
                        "range": 1.0,
                    }
            except (ValueError, TypeError):
                # Structural safety tag defining a non-numeric/categorical column
                feature_variance_bounds[feature] = None

        # Calculate generalized distance discrepancy score parameters
        scored_profiles = []
        for profile in retrieved_candidates:
            discrepancy_penalty = 0.0

            for feature in self.tracking_features:
                ideal_target = target_metrics_dict.get(feature)
                observed_value = profile.get(feature)

                if (
                    ideal_target is not None
                    and observed_value is not None
                    and observed_value != ""
                ):
                    bounds = feature_variance_bounds[feature]

                    if bounds is not None:
                        # 1. Evaluate as a scaled continuous physical metric
                        try:
                            normalized_observed = (
                                float(observed_value) - bounds["minimum"]
                            ) / bounds["range"]
                            normalized_target = (
                                float(ideal_target) - bounds["minimum"]
                            ) / bounds["range"]
                            discrepancy_penalty += (
                                normalized_observed - normalized_target
                            ) ** 2
                        except (ValueError, TypeError):
                            discrepancy_penalty += 1.0
                    else:
                        # 2. Evaluate as an explicit categorical string mapping match
                        observed_str = str(observed_value).strip().lower()
                        target_str = str(ideal_target).strip().lower()
                        if observed_str != target_str:
                            discrepancy_penalty += 1.0

            scored_profiles.append((discrepancy_penalty, profile))

        # Sort based on lowest localized spatial error vectors
        scored_profiles.sort(key=lambda profile_tuple: profile_tuple[0])
        optimized_matches = [
            profile for _, profile in scored_profiles[:top_matches_count]
        ]

        print(
            f"[Retriever] Successfully finalized {top_matches_count} dynamic baseline matches.",
            flush=True,
        )
        return optimized_matches

### Define Synthetic Generator agent

In [114]:
DEFAULT_SYSTEM_INSTRUCTION_TEMPLATE = (
    "Role. You are a precise synthetic data scientist and automated schema matrix generator. "
    "You must output exactly ONE new synthetic row that matches the behavioral criteria and patterns of the requested scenario.\n\n"
    "REQUIRED SCHEMA STRUCTURE:\n"
    "Your output JSON must contain EVERY single column key listed below. Do not omit any key.\n"
    "{anchor_data_text}\n\n"
    "CRITICAL OUTPUT INSTRUCTION:\n"
    "Output ONLY a valid JSON object. Do not include markdown blocks, ticks (```json), or commentary. "
    "Every numerical variable must be populated with a valid, non-empty float or integer value based on the target scenario.\n\n"
    "HISTORICAL SYSTEM CONTEXT CORRELATIONS:\n{context_records_json}\n\n"
    "TARGET SCENARIO TO SYNTHESIZE:\n\"{scenario_prompt}\"\n\n"
    "Output your single complete structural JSON object row now:"
)


class SyntheticGenerator:
    def __init__(self, llm_url, model_id, dataframe=None, feature_columns=None, prompt_template=None):
        self.api_url = llm_url
        self.model_id = model_id
        self.prompt_template = prompt_template if prompt_template is not None else DEFAULT_SYSTEM_INSTRUCTION_TEMPLATE
        
        if feature_columns is not None:
            self.tracking_features = feature_columns
        elif dataframe is not None and isinstance(dataframe, pd.DataFrame):
            self.tracking_features = list(dataframe.columns)
        else:
            self.tracking_features = []

        self.anchor_data_text = ", ".join([f'"{col}": <value>' for col in self.tracking_features])

    def _sanitize_records(self, records):
        """Replaces empty string artifacts with neutral zeros or removes them to prevent truncation."""
        sanitized = []
        for rec in records:
            clean_rec = {}
            for k, v in rec.items():
                if v == '' or v is None:
                    clean_rec[k] = 0.0 
                else:
                    clean_rec[k] = v
            sanitized.append(clean_rec)
        return sanitized

    def generate_synthetic_row(self, scenario_prompt, matching_records):
        """
        Passes the scenario, sanitized context records, and strict structural requirements
        to the model, ensuring a fully populated data row is generated.
        """
        sanitized_records = self._sanitize_records(matching_records)
        context_records_json = json.dumps(sanitized_records, indent=2)

        system_instruction = self.prompt_template.format(
            anchor_data_text=self.anchor_data_text,
            context_records_json=context_records_json,
            scenario_prompt=scenario_prompt
        )

        payload = {
            "model": self.model_id,
            "prompt": system_instruction,
            "stream": False,
            "format": "json",
            "options": {
                "temperature": 0.3,
                "num_ctx": 16384
            }
        }

        raw_output = ""
        try:
            response = requests.post(self.api_url, json=payload, timeout=300)
            response.raise_for_status()
            
            result_json = response.json()
            raw_output = result_json.get("response", "").strip()
            
            if not raw_output and "thinking" in result_json:
                raw_output = result_json.get("thinking", "").strip()
            
            cleaned_text = re.sub(r'^```[a-zA-Z]*\s*|\s*```$', '', raw_output).strip()
            
            if not (cleaned_text.startswith('{') and cleaned_text.endswith('}')):
                match = re.search(r'(\{.*\})', cleaned_text, re.DOTALL)
                if match:
                    cleaned_text = match.group(1).strip()
                    
            parsed_json = json.loads(cleaned_text)
            
            # Post-processing verification: Ensure no dropped columns map back as missing
            for feature in self.tracking_features:
                if feature not in parsed_json:
                    parsed_json[feature] = 0.0
                    
            return parsed_json
            
        except Exception as e:
            print(f"[Generator] Error generating or parsing structural synthetic row: {e}")
            print(f"Raw model response was: {raw_output}")
            raise e

### Define Neural Network Validator with helper functions

In [115]:
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler

class FullAutoencoder(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 24), nn.ReLU(),
            nn.Linear(24, 8), nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.Linear(8, 24), nn.ReLU(),
            nn.Linear(24, input_dim)
        )
    def forward(self, x): 
        return self.decoder(self.encoder(x))

class NNValidator:
    def __init__(self, epochs=60, batch_size=32, lr=0.01):
        self.epochs = epochs
        self.batch_size = batch_size
        self.lr = lr
        self.scaler = StandardScaler()
        self.model = None
        self.feature_columns = []
        self.feature_medians_ = {}
        self.threshold = 0.0
        self.loss_weights_tensor = None
        self.importance_weights = {} 

    def _clean_and_prepare(self, df_raw, is_training=True):
        df_cleaned = df_raw.iloc[1:].copy() if df_raw.iloc[0].astype(str).str.contains(r'\[.*\]').any() else df_raw.copy()
        numeric_cols = [col for col in df_cleaned.columns if col not in ['Club', 'Ball']]
        
        for col in numeric_cols:
            df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')
            if is_training:
                self.feature_medians_[col] = df_cleaned[col].median()
        
        if is_training:
            self.feature_columns = numeric_cols
            weights_dict = self.importance_weights or {}
            weights_list = [weights_dict.get(col, 1.0) for col in self.feature_columns]
            self.loss_weights_tensor = torch.tensor(weights_list, dtype=torch.float32)

        return df_cleaned[self.feature_columns].fillna(self.feature_medians_)

    def train_with_weights(self, df_historical_raw, optimized_weights, quiet=True):
        self.importance_weights = optimized_weights
        X_df = self._clean_and_prepare(df_historical_raw, is_training=True)
        X_scaled = self.scaler.fit_transform(X_df)
        X_tensor = torch.tensor(X_scaled, dtype=torch.float32)

        self.model = FullAutoencoder(X_tensor.shape[1])
        optimizer = optim.Adam(self.model.parameters(), lr=self.lr)

        self.model.train()
        dataset_size = len(X_tensor)
        for epoch in range(self.epochs):
            permutation = torch.randperm(dataset_size)
            for i in range(0, dataset_size, self.batch_size):
                indices = permutation[i:i+self.batch_size]
                batch_x = X_tensor[indices]

                outputs = self.model(batch_x)
                loss = torch.mean(((outputs - batch_x) ** 2) * self.loss_weights_tensor.to(batch_x.device))

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        self.model.eval()
        with torch.no_grad():
            train_predictions = self.model(X_tensor)
            weighted_mse = torch.mean(((X_tensor - train_predictions) ** 2) * self.loss_weights_tensor, dim=1).numpy()
            self.threshold = np.percentile(weighted_mse, 95) + 0.2
        
        if not quiet:
            print(f"[NN Production] Model ready. Separation threshold: {self.threshold:.4f}")

    def validate_row(self, synthetic_row_dict):
        if self.model is None: 
            raise ValueError("Train model first.")
        try:
            self.model.eval()
            row_vector = []
            for col in self.feature_columns:
                val = synthetic_row_dict.get(col, None)
                if val is None or pd.isna(val) or str(val).strip() == "":
                    val = self.feature_medians_.get(col, 0.0)
                row_vector.append(float(val))
            
            row_df = pd.DataFrame([row_vector], columns=self.feature_columns)
            X_tensor = torch.tensor(self.scaler.transform(row_df), dtype=torch.float32)

            with torch.no_grad():
                reconstruction = self.model(X_tensor)
                row_mse = torch.mean(((X_tensor - reconstruction) ** 2) * self.loss_weights_tensor).item()
            
            if np.isnan(row_mse):
                return False
            return row_mse <= self.threshold
        except Exception as e:
            print(f"[NN Validator Error] {e}")
            return False

In [ ]:
def compile_suite_to_tensor(suite, base_validator):
    """Transforms a collection of synthetic candidate record dictionaries into a fully 

    sanitized, scaled, and aligned PyTorch tensor optimized for neural manifold evaluation.

    Performs feature-order synchronization, dynamic median imputation for missing values,
    and standard deviation scaling to match the neural network's trained input dimensions.
    """
    matrix = []
    for sample in suite:
        row_features = []
        for col in base_validator.feature_columns:
            val = sample.get(col, None)
            if val is None or pd.isna(val) or str(val).strip() == "":
                val = base_validator.feature_medians_.get(col, 0.0)
            row_features.append(float(val))
        matrix.append(row_features)

    matrix_df = pd.DataFrame(matrix, columns=base_validator.feature_columns)
    return torch.tensor(base_validator.scaler.transform(matrix_df), dtype=torch.float32)

In [ ]:
def optimize_pipeline_weights(df_raw, valid_suite, invalid_suite, iterations=200):
    """Executes a contrastive meta-optimization routine over the manifold loss weights matrix.

    This function automatically scales the error multipliers for each physical feature
    to maximize the separation gap between valid inliers and invalid physics violations, 
    making the downstream Judge highly sensitive to specific structural flaws.
    """
    print(f"--- Meta-Optimizer Running ({iterations} epochs) ---")
    
    base_validator = NNValidator(epochs=35, batch_size=32, lr=0.01)
    base_validator.train_with_weights(df_raw, optimized_weights=None, quiet=True)
    base_validator.model.eval()
    
    valid_tensors = compile_suite_to_tensor(valid_suite, base_validator)
    invalid_tensors = compile_suite_to_tensor(invalid_suite, base_validator)
    
    with torch.no_grad():
        valid_base_errors = (valid_tensors - base_validator.model(valid_tensors)) ** 2
        invalid_base_errors = (invalid_tensors - base_validator.model(invalid_tensors)) ** 2

    meta_weights = torch.nn.Parameter(torch.ones(len(base_validator.feature_columns), dtype=torch.float32))
    meta_optimizer = optim.Adam([meta_weights], lr=0.01)

    for epoch in range(iterations):
        meta_optimizer.zero_grad()
        bounded_multipliers = torch.clamp(meta_weights, min=1.0, max=100.0)
        
        mean_valid_loss = torch.mean(torch.mean(valid_base_errors * bounded_multipliers, dim=1))
        soft_min_invalid_loss = 1.0 / torch.mean(1.0 / (torch.mean(invalid_base_errors * bounded_multipliers, dim=1) + 1e-4))
        
        contrast_loss = - (soft_min_invalid_loss / (mean_valid_loss + 1e-6))
        contrast_loss.backward()

        torch.nn.utils.clip_grad_norm_([meta_weights], max_norm=1.0)
        meta_optimizer.step()
        
        if (epoch + 1) % 50 == 0 or epoch == 0:
            print(f"  Meta-Epoch {epoch + 1:03d} -> Contrast Gap: {abs(contrast_loss.item()):.2f}x")

    final_multipliers = torch.clamp(meta_weights, min=1.0, max=100.0).detach().numpy()
    return {col: float(final_multipliers[i]) for i, col in enumerate(base_validator.feature_columns)}

### Define Distribution Validator

In [ ]:
from sklearn.ensemble import IsolationForest

class DistributionValidator:
    """A pure telemetry utility that establishes statistical baselines and calculates 

    high-dimensional data density footprints across telemetry payloads.

    Combines independent univariate tracking metrics (Z-scores, soft min/max boundaries) 
    with an unsupervised Isolation Forest model to detect multivariate anomaly signatures.
    """
    def __init__(self, contamination=0.01):
        self.contamination = contamination
        self.iso_forest = None
        self.feature_cols = []
        self.feature_bounds = {}
        self.is_fitted = False
        
    def fit(self, df_baseline, feature_cols):
        self.feature_cols = list(feature_cols)
        valid_cols = [c for c in self.feature_cols if c in df_baseline.columns]
        if not valid_cols:
            raise KeyError("None of the specified feature columns were found in the dataset.")
        self.feature_cols = valid_cols
        
        df_clean = df_baseline[self.feature_cols].copy().replace([np.inf, -np.inf], np.nan).dropna()
        if len(df_clean) < 50:
            raise ValueError(f"Baseline rows count ({len(df_clean)}) insufficient for training.")
            
        for col in self.feature_cols:
            self.feature_bounds[col] = {
                'min': float(df_clean[col].min()),
                'max': float(df_clean[col].max()),
                'mean': float(df_clean[col].mean()),
                'std': float(df_clean[col].std() if df_clean[col].std() > 0 else 1.0)
            }
            
        self.iso_forest = IsolationForest(contamination=self.contamination, random_state=42, n_estimators=150)
        self.iso_forest.fit(df_clean)
        self.is_fitted = True
        print(f"--> DistributionValidator successfully trained on {len(df_clean)} rows.")
        
    def extract_telemetry(self, shot_payload):
        """Outputs raw statistical tracking metrics without rendering a pass/fail verdict."""
        if not self.is_fitted:
            raise RuntimeError("Validator must be fitted.")
            
        row_data = {}
        for col in self.feature_cols:
            fallback_val = self.feature_bounds[col]['mean']
            val = shot_payload.get(col, None)
            if val is None or pd.isna(val):
                val = fallback_val
            try:
                row_data[col] = float(val)
            except (TypeError, ValueError):
                row_data[col] = float(fallback_val)
            
        df_single = pd.DataFrame([row_data])[self.feature_cols]
        
        z_scores = {}
        boundary_breaches = {}
        for col in self.feature_cols:
            val = row_data[col]
            b = self.feature_bounds[col]
            
            z_scores[col] = abs(val - b['mean']) / b['std']
            
            lower_tolerance = b['min'] - (0.10 * abs(b['min']))
            upper_tolerance = b['max'] + (0.10 * abs(b['max']))
            boundary_breaches[col] = (val < lower_tolerance or val > upper_tolerance)

        multivariate_score = float(self.iso_forest.decision_function(df_single)[0])
        is_iForest_inlier = int(self.iso_forest.predict(df_single)[0]) == 1
        
        return {
            "row_values": row_data,
            "z_scores": z_scores,
            "boundary_breaches": boundary_breaches,
            "multivariate_score": multivariate_score,
            "is_inlier": is_iForest_inlier
        }

In [119]:
def map_payload_to_feature_cols(payload, target_features):
    """
    Dynamically aligns short payload keys with the corresponding 
    long keys in feature_cols using normalized substring matching.
    """
    def normalize(text):
        # Convert to lowercase and strip common trailing symbols/units
        return str(text).lower().replace("(mph)", "").replace("-", " ").strip()

    mapped_payload = {}
    target_norm_map = {normalize(col): col for col in target_features}

    for payload_key, val in payload.items():
        norm_p_key = normalize(payload_key)

        matched_col = target_norm_map.get(norm_p_key)
        if matched_col is None:
            for col in target_features:
                norm_col = normalize(col)
                if norm_p_key in norm_col or norm_col in norm_p_key:
                    matched_col = col
                    break

        if matched_col:
            mapped_payload[matched_col] = val

    return mapped_payload

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

class IsolationForestValidator:
    """An advanced multivariate statistical validation engine that builds an unsupervised 

    isolation forest manifold over normalized data structures.

    Handles categorical exclusion, automated column type casting, median imputation for
    missing values, and robust standard scaling before fitting the outlier detection ensemble.
    """
    def __init__(self, contamination=0.05, n_estimators=150, random_state=42):
        self.contamination = contamination
        self.n_estimators = n_estimators
        self.random_state = random_state
        self.scaler = StandardScaler()
        self.model = None
        self.feature_columns = []
        self.feature_medians_ = {}
        self.threshold = 0.0

    def _clean_and_prepare(self, df_raw, is_training=True):
        df_cleaned = df_raw.copy()
        numeric_cols = [col for col in df_cleaned.columns if col not in ['Club', 'Ball']]
        
        for col in numeric_cols:
            df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')
            if is_training:
                self.feature_medians_[col] = df_cleaned[col].median()
        
        if is_training:
            self.feature_columns = numeric_cols

        return df_cleaned[self.feature_columns].fillna(self.feature_medians_)

    def train(self, df_historical_raw, quiet=False):
        X_df = self._clean_and_prepare(df_historical_raw, is_training=True)
        X_scaled = self.scaler.fit_transform(X_df)
        
        self.model = IsolationForest(
            n_estimators=self.n_estimators,
            contamination=self.contamination,
            random_state=self.random_state,
            n_jobs=-1
        )
        self.model.fit(X_scaled)
        
        train_scores = self.model.score_samples(X_scaled)
        self.threshold = np.percentile(train_scores, self.contamination * 100)
        
        if not quiet:
            print(f"[iForest Production] Model fit complete across {X_scaled.shape[0]} rows.")
            print(f"[iForest Production] Separation manifold score threshold: {self.threshold:.4f}")

    def validate_row(self, synthetic_row_dict):
        if self.model is None: 
            raise ValueError("Train model first.")
        try:
            row_vector = []
            for col in self.feature_columns:
                val = synthetic_row_dict.get(col, None)
                if val is None or pd.isna(val) or str(val).strip() == "":
                    val = self.feature_medians_.get(col, 0.0)
                row_vector.append(float(val))
            
            row_df = pd.DataFrame([row_vector], columns=self.feature_columns)
            X_scaled = self.scaler.transform(row_df)
            row_score = self.model.score_samples(X_scaled)[0]
            
            if np.isnan(row_score):
                return False
                
            return row_score >= self.threshold
        except Exception as e:
            print(f"[iForest Validator Error] {e}")
            return False

### Define Judge agent

In [121]:
DEFAULT_JUDGE_PROMPT_TEMPLATE = """
You are an expert Data Engineer and Mathematical Validation Judge for a high-dimensional synthetic data generation loop.

Your role is to evaluate a proposed data point alongside a raw mathematical telemetry JSON block and decide if the data point represents a structurally and logically consistent real-world scenario or a corrupted generation artifact.

### PROPOSED SYNTHETIC DATAPOINT PAYLOAD:
{candidate_payload_json}

### COMPREHENSIVE MATHEMATICAL TELEMETRY:
{telemetry_package_json}

---

### HOW TO READ AND USE THE TELEMETRY FOR YOUR ADJUDICATION:

1. **"pipeline_alerts" (Hard Violations):**
   * These are immediate, rule-based mathematical or logical boundary violations. If you see alerts here, the proposed data has breached known statistical or structural constraints.

2. **"distribution_validator" (Statistical Alignment):**
   * **"z_scores":** Indicates how many standard deviations ($\sigma$) a value lies away from the historical average.
     * Values between $0.0\sigma$ and $2.0\sigma$ are perfectly normal.
     * Values between $2.0\sigma$ and $3.5\sigma$ represent rare, extreme, but possible edge cases.
     * Values $> 3.5\sigma$ are severe outliers. Identify which parameter is excessive and direct the generator to scale it back.
   * **"boundary_breaches":** If a parameter registers `true` here, it has broken past absolute maximum or minimum bounds recorded in real data.
   * **"is_inlier" (Multivariate Co-dependency):** If this is `false`, individual features might look normal in isolation, but their *multivariate relationship* or ratio is physically/logically impossible.

3. **"neural_network_manifold" (Manifold Consistency):**
   * **"global_weighted_mse":** The overall error representing how well our neural network autoencoder could reconstruct this specific sample. A value higher than the threshold limit implies a broken manifold geometry.
   * **"feature_squared_losses":** Diagnostic feature mapping. Features with abnormally high losses pinpoint exactly which specific column caused the contradiction.

---

### CRITICAL ADJUDICATION INSTRUCTIONS:
1. Identify high-loss nodes in `feature_squared_losses` and high deviations in `z_scores` to pinpoint exactly which variables are out of alignment.
2. If statistical alerts are clean but you detect an implicit physical/logical correlation violation that the automated models missed, you must **Reject** the data point (`approved: false`).
3. If the mathematical models flagged a minor anomaly, but you determine that the data represents an acceptable real-world edge case, you may **Accept** it (`approved: true`).
4. When rejecting, explain the contradictory relationships clearly. **DO NOT propose explicit, direct replacement numerical values**. Instead, describe the parameters causing the tension and dictate the *directional* shift or structural constraint required.

You must format your final response strictly as a single, valid, raw JSON object matching the schema template below.

EXPECTED JSON OUTPUT SCHEMA:
{{
    "approved": true/false,
    "physics_flaws": ["List of precise technical descriptions detailing any broken data parameters or desynchronized correlations discovered."],
    "recalibration_instructions": "Clear, mathematically explicit feedback instructing the challenger agent exactly how to alter its boundaries or generation equations to pass next time. Leave empty if approved."
}}
"""

class Judge:
    def __init__(
        self,
        dist_validator,
        nn_validator,
        llm_url,
        model_id,
        z_threshold,
        prompt_template=None,
    ):
        self.dist_validator = dist_validator
        self.nn_validator = nn_validator
        self.api_url = llm_url
        self.model_id = model_id
        self.z_threshold = z_threshold
        
        # Uses custom prompt if passed in, otherwise falls back to the new default
        self.prompt_template = (
            prompt_template
            if prompt_template is not None
            else DEFAULT_JUDGE_PROMPT_TEMPLATE
        )

    def _evaluate_nn(self, candidate_payload):
        if self.nn_validator.model is None:
            raise ValueError(
                "The provided Neural Network model has not been trained yet."
            )

        try:
            self.nn_validator.model.eval()
            row_vector = []
            for col in self.nn_validator.feature_columns:
                fallback_val = self.nn_validator.feature_medians_.get(col, 0.0)
                val = candidate_payload.get(col, None)
                if (
                    val is None
                    or pd.isna(val)
                    or str(val).strip() == ""
                ):
                    val = fallback_val
                try:
                    row_vector.append(float(val))
                except (TypeError, ValueError):
                    row_vector.append(float(fallback_val))

            row_df = pd.DataFrame(
                [row_vector], columns=self.nn_validator.feature_columns
            )
            X_tensor = torch.tensor(
                self.nn_validator.scaler.transform(row_df), dtype=torch.float32
            )

            with torch.no_grad():
                reconstruction = self.nn_validator.model(X_tensor)

                squared_errors = (X_tensor - reconstruction) ** 2
                weighted_errors = (
                    squared_errors * self.nn_validator.loss_weights_tensor
                )
                row_mse = torch.mean(weighted_errors).item()

            is_nn_valid = (not np.isnan(row_mse)) and (
                row_mse <= self.nn_validator.threshold
            )

            nn_feature_errors = {}
            weighted_err_np = weighted_errors.squeeze().cpu().numpy()
            for idx, col in enumerate(self.nn_validator.feature_columns):
                val_err = (
                    weighted_err_np[idx]
                    if weighted_err_np.ndim > 0
                    else float(weighted_err_np)
                )
                nn_feature_errors[col] = float(val_err)

            return is_nn_valid, row_mse, nn_feature_errors

        except Exception as e:
            print(f"[Judge -> NN Internal Check Error] {e}")
            return False, float("inf"), {}

    def _build_llm_prompt(self, candidate_payload, telemetry):
        json_fallback = (
            lambda o: (
                o.item()
                if hasattr(o, "item") and callable(getattr(o, "item"))
                else str(o)
            )
        )

        return self.prompt_template.format(
            candidate_payload_json=json.dumps(
                candidate_payload, indent=2, default=json_fallback
            ),
            telemetry_package_json=json.dumps(
                telemetry, indent=2, default=json_fallback
            ),
        )

    def evaluate_generation(self, candidate_payload):
        """Evaluates row against NN and Isolation Forest, routing complete data through

        Ollama for a final verdict only if anomalies are present.
        """

        # 1. Gather comprehensive telemetry maps from the distribution tools
        dist_metrics = self.dist_validator.extract_telemetry(candidate_payload)

        # 2. Gather feature-level error maps from the Neural Network tool
        nn_valid, nn_mse, nn_feature_errors = self._evaluate_nn(
            candidate_payload
        )

        # 3. Process automated anomalies as initial context hints
        deterministic_failures = []
        for col, z in dist_metrics["z_scores"].items():
            if z > self.z_threshold:
                deterministic_failures.append(
                    f"Univariate Alert: '{col}' is a statistical outlier"
                    f" ({z:.2f}σ from baseline mean)."
                )
            if dist_metrics["boundary_breaches"][col]:
                deterministic_failures.append(
                    f"Data-Wall Alert: '{col}' completely breached empirical"
                    " absolute limits."
                )

        if not dist_metrics["is_inlier"]:
            deterministic_failures.append(
                "Multivariate Alert: Isolation Forest flagged anomalous"
                " cross-feature co-dependencies."
            )

        if not nn_valid:
            deterministic_failures.append(
                "Neural Network Alert: Weighted reconstruction loss"
                f" ({nn_mse:.4f}) exceeded the target manifold limit"
                f" ({self.nn_validator.threshold:.4f})."
            )

        # 4. Package full-system raw telemetry footprint
        full_telemetry_package = {
            "pipeline_alerts": deterministic_failures,
            "distribution_validator": dist_metrics,
            "neural_network_manifold": {
                "passed_baseline": nn_valid,
                "global_weighted_mse": nn_mse,
                "feature_squared_losses": nn_feature_errors,
            },
        }

        max_z = (
            max(dist_metrics["z_scores"].values())
            if dist_metrics["z_scores"]
            else 0.0
        )

        # 5. FAST-PATH EARLY EXIT (BYPASS LLM CALL FOR PRISTINE DATA POINTS)
        if nn_valid and len(deterministic_failures) == 0 and dist_metrics["is_inlier"]:
            feedback = {
                "approved": True,
                "metrics": {
                    "iForest_score": dist_metrics["multivariate_score"],
                    "nn_mse": nn_mse,
                    "max_z_score": max_z,
                },
                "critique_text": "Success: Row aligns flawlessly with historical manifold distribution.",
                "raw_telemetry": full_telemetry_package,
            }
            return True, feedback

        # 6. ADVERSARIAL LLM CALL (TRIGGERED ONLY ON ANOMALIES/VIOLATIONS)
        prompt = self._build_llm_prompt(candidate_payload, full_telemetry_package)

        payload = {
            "model": self.model_id,
            "prompt": prompt,
            "stream": False,
            "format": "json",
            "options": {"temperature": 0.0},
        }

        print(
            f"[hybrid_judge] Anomalies detected! Processing multi-model"
            f" telemetry suite via Ollama ({self.model_id})...",
            flush=True,
        )
        raw_response_text = ""
        try:
            response = requests.post(self.api_url, json=payload, timeout=300)
            response.raise_for_status()

            result_json = response.json()

            raw_response_text = result_json.get("response", "").strip()
            if not raw_response_text and "thinking" in result_json:
                raw_response_text = result_json.get("thinking", "").strip()

            cleaned_text = re.sub(
                r"^```[a-zA-Z]*\s*|\s*```$", "", raw_response_text
            ).strip()

            if not (cleaned_text.startswith("{") and cleaned_text.endswith("}")):
                match = re.search(r"(\{.*\})", cleaned_text, re.DOTALL)
                if match:
                    cleaned_text = match.group(1).strip()

            verdict_data = json.loads(cleaned_text)
            is_approved = bool(verdict_data.get("approved", False))
            physics_flaws = verdict_data.get("physics_flaws", [])
            recalibration_instructions = verdict_data.get(
                "recalibration_instructions", ""
            )

        except (
            requests.exceptions.RequestException,
            json.JSONDecodeError,
            KeyError,
        ) as e:
            print(
                f"[hybrid_judge] Interface error encountered: {e}. Reverting"
                " to rigid rule-based logic.",
                flush=True,
            )
            math_rules_passed = (len(deterministic_failures) == 0) and nn_valid
            return math_rules_passed, {
                "approved": math_rules_passed,
                "metrics": {
                    "iForest_score": dist_metrics["multivariate_score"],
                    "nn_mse": nn_mse,
                    "max_z_score": max_z,
                },
                "critique_text": (
                    f"Fallback Critique: Automated check failure. Logs:"
                    f" {', '.join(deterministic_failures)}"
                    if not math_rules_passed
                    else "Success."
                ),
                "raw_telemetry": full_telemetry_package,
            }

        feedback = {
            "approved": is_approved,
            "metrics": {
                "iForest_score": dist_metrics["multivariate_score"],
                "nn_mse": nn_mse,
                "max_z_score": max_z,
            },
            "critique_text": "",
            "raw_telemetry": full_telemetry_package,
        }

        if is_approved:
            feedback["critique_text"] = (
                "Success: Row aligns flawlessly with historical manifold distribution."
            )
        else:
            critique = "=== AGENTIC JUDGE CORRECTION CRITIQUE ===\n"
            critique += "Your proposed synthetic data point has been REJECTED for structural flaws:\n"
            for flaw in physics_flaws:
                critique += f"  * {flaw}\n"
            critique += (
                f"\n### TARGETED INSTRUCTIONS FOR NEXT"
                f" ITERATION:\n{recalibration_instructions}"
            )
            feedback["critique_text"] = critique

        return is_approved, feedback

<>:20: SyntaxWarning: invalid escape sequence '\s'
<>:20: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_2178/505674238.py:20: SyntaxWarning: invalid escape sequence '\s'
  * **"z_scores":** Indicates how many standard deviations ($\sigma$) a value lies away from the historical average.


### Define Main Ochestrator agent with tools: challenger, retriever, generator, judge

In [122]:
import time

class MainOrchestrator:
    def __init__(self, challenger, retriever, generator, judge):
        """
        Coordinates the self-correcting loop between the brainstorming Challenger,
        the Hybrid Retriever, the Synthetic Generator, and the Multi-Manifold Judge.
        """
        self.challenger = challenger
        self.retriever = retriever  
        self.generator = generator
        self.judge = judge

    def run_generation_pipeline(self, max_iterations=8):
        """
        Executes the closed-loop optimization cycle. Returns the approved synthetic record
        or logs a failure payload if it exceeds the iteration ceiling.
        """
        previous_failures = []
        failure_mode = None
        
        print("=" * 80)
        print("INITIALIZING SELF-CORRECTING DATA ORCHESTRATION LOOP")
        print("=" * 80)
        
        start_time = time.time()
        
        for iteration in range(1, max_iterations + 1):
            print(f"\n[ITERATION {iteration}/{max_iterations}]")
            print("-" * 40)
            
            # Step 1: Brainstorm or adjust a scenario via the Challenger
            challenger_result = self.challenger.generate(
                previous_failures=previous_failures, 
                failure_mode=failure_mode
            )
            if challenger_result is None:
                print("⚠️ [Orchestrator] Challenger returned None. Retrying...")
                failure_mode = "Previous generation failed to produce valid structured JSON output."
                continue
            scenario, target_numerical_dict = challenger_result
            print(f"[Orchestrator -> Challenger] Targeting Scenario:\n \"{scenario}\"\n")
            print(f"[Orchestrator -> Challenger] Targeting Profile Matrix:\n {json.dumps(target_numerical_dict)}")
            
            # Step 2: Extract real physical records using Hybrid Reranking (Text Vibe + Strict Math Spacing)
            print("[Orchestrator -> Retriever] Querying context from Hybrid Retriever (ChromaDB + Local L2 Reranking)...")
            
            # === FIXED: Explicitly using parameters matching your Retriever class definition ===
            context_records = self.retriever.retrieve(
                scenario_description=scenario, 
                target_metrics_dict=target_numerical_dict,
                semantic_pool_size=30,  
                top_matches_count=3     
            )
            
            print("\n🔍 [LOG] HISTORICAL EXAMPLES RETRIEVED FROM VECTOR MEMORY:")
            print(json.dumps(context_records, indent=2))
            print("-" * 40 + "\n")
            
            # Step 3: Pass instructions and context rows to the Generator
            print("[Orchestrator -> Generator] Dispatching constraints to generator node...")
            try:
                candidate_payload = self.generator.generate_synthetic_row(scenario, context_records)
                
                print("\n✨ [LOG] NEW RAW SYNTHETIC EXAMPLE GENERATED BY LLM:")
                print(json.dumps(candidate_payload, indent=2))
                print("-" * 40 + "\n")
                
            except Exception as e:
                print(f"⚠️ [Orchestrator Warning] Structural parsing error on generation layout: {e}")
                previous_failures.append(scenario)
                failure_mode = "JSON structural formatting failure. Your response could not be parsed."
                continue
                
            # Step 4: Dynamically map incoming keys to our mathematical feature matrix column names
            aligned_payload = map_payload_to_feature_cols(
                candidate_payload, 
                self.judge.dist_validator.feature_cols
            )
            
            # Step 5: Send the aligned tracking payload through the Judge
            print("[Orchestrator -> Judge] Evaluating row against isolation forests and neural manifolds...")
            is_approved, report = self.judge.evaluate_generation(aligned_payload)
            
            # --- PRINT METRIC RESULTS FROM NN AND DISTRIBUTION VALIDATION ---
            print("\n [LOG] MATHEMATICAL VALIDATION METRICS:")
            print(f"  * [Distribution Validation] Isolation Forest Score: {report['metrics']['iForest_score']:.4f}")
            print(f"  * [Distribution Validation] Max Feature Z-Score:      {report['metrics']['max_z_score']:.2f}σ")
            
            nn_telemetry = report["raw_telemetry"]["neural_network_manifold"]
            print(f"  * [Neural Network Manifold] Weighted Row MSE:        {nn_telemetry['global_weighted_mse']:.4f}")
            print(f"  * [Neural Network Manifold] Target Threshold Limit:  {self.judge.nn_validator.threshold:.4f}")
            print(f"  * [Neural Network Manifold] Manifold System Passed:  {nn_telemetry['passed_baseline']}")
            print("-" * 40 + "\n")
            
            # Step 6: Route state logic based on evaluation verdict
            if is_approved:
                elapsed_time = time.time() - start_time
                print("\n" + "=" * 80)
                print(f"🎉 PIPELINE SUCCESS: Valid synthetic data point secured in {iteration} iteration(s)!")
                print(f"⏱️ Total processing duration: {elapsed_time:.2f} seconds")
                print("=" * 80)
                
                return {
                    "status": "SUCCESS",
                    "iterations_used": iteration,
                    "synthetic_record": aligned_payload,
                    "telemetry_metrics": report["metrics"],
                    "raw_telemetry": report["raw_telemetry"]
                }
            else:
                print(f"❌ JUDGE REJECTED: Candidate failed manifold constraints.")
                previous_failures.append(scenario)
                failure_mode = report["critique_text"]
                print(f"[Orchestrator] Feedback captured and locked down for next loop.")
                
        elapsed_time = time.time() - start_time
        print("\n" + "!" * 80)
        print(f"⚠️ PIPELINE ABORTED: Max limit of {max_iterations} iterations reached without meeting validation standards.")
        print(f"⏱️ Expired after {elapsed_time:.2f} seconds.")
        print("!" * 80)
        
        return {
            "status": "TIMEOUT_FAILURE",
            "iterations_used": max_iterations,
            "synthetic_record": None,
            "latest_critique": failure_mode,
            "raw_telemetry": report["raw_telemetry"] if 'report' in locals() else None
        }

In [123]:
# preprocess data
from sklearn.model_selection import train_test_split

print("--- Starting production data preprocessing ---")

# Find non-numeric/metadata columns programmatically
metadata_columns = set(df.select_dtypes(exclude=[np.number]).columns)

# Pull model inputs directly from the dataframe and exclude discovered metadata columns
tracking_feature_columns = [
    column for column in df.columns if column not in metadata_columns
]
feature_cols = tracking_feature_columns

print(
    f"[Preprocessing] Programmatically discovered {len(tracking_feature_columns)} numeric feature columns."
)
print(f"[Preprocessing] Dynamically excluding non-numeric metadata columns: {sorted(list(metadata_columns))}")

# Coerce numeric features and clean infinite artifacts
for feature_name in tracking_feature_columns:
    df[feature_name] = pd.to_numeric(df[feature_name], errors="coerce")

print(
    "[Preprocessing] Converted feature columns to numeric values and marked"
    " invalid entries as NaN."
)
df = df.replace([np.inf, -np.inf], np.nan)

# 4. Fill missing values with a per-column median
global_medians = {
    feature_name: float(
        df[feature_name].median() if not df[feature_name].isna().all() else 0.0
    )
    for feature_name in tracking_feature_columns
}
for feature_name, median_value in global_medians.items():
    df[feature_name] = df[feature_name].fillna(median_value)

print(
    f"[Preprocessing] Parsed {len(tracking_feature_columns)} metrics across"
    f" {len(df)} rows."
)

--- Starting production data preprocessing ---
[Preprocessing] Programmatically discovered 33 numeric feature columns.
[Preprocessing] Dynamically excluding non-numeric metadata columns: ['Ball', 'Club']
[Preprocessing] Converted feature columns to numeric values and marked invalid entries as NaN.
[Preprocessing] Parsed 33 metrics across 10169 rows.


### Split data

In [ ]:
target_column = 'Total'
feature_columns = [column for column in tracking_feature_columns if column != target_column]
TARGET_COL = target_column
FEATURE_COLS = feature_columns

# Keep the final 15% completely untouched until evaluation time.
train_pool_df, isolated_test_df = train_test_split(df, test_size=0.15, random_state=42)
# Split the remaining pool again so the NN sees a separate internal holdout set.
train_df, holdout_df = train_test_split(train_pool_df, test_size=0.20, random_state=42)

print("=" * 60)
print("LEAK-PROOF HIERARCHICAL SPLIT COMPLETE")
print(f"   * Total Dataset Base:          {len(df)} rows")
print(f"   * Real Validation Pool (85%):  {len(train_pool_df)} rows")
print(f"     ├── NN Training Split:       {len(train_df)} rows")
print(f"     └── NN Holdout Validation:   {len(holdout_df)} rows")
print(f"   * Isolated Evaluation Test Set: {len(isolated_test_df)} rows (15% Untouchable)")
print("=" * 60)
print("[Split] The isolated test set is not used anywhere else in this cell.")

# =====================================================================
# PHASE 3: VALIDATION SUITE SAMPLING
# =====================================================================
print("\n Initiating multi-tiered validation suite sampling...")
print("[Sampling] Training an isolation forest on the internal training split.")

anomaly_validator = IsolationForestValidator(contamination=0.15, n_estimators=200)
anomaly_validator.train(train_df, quiet=True)

prepared_train_features = anomaly_validator._clean_and_prepare(train_df, is_training=False)
scaled_train_features = anomaly_validator.scaler.transform(prepared_train_features)

scored_train_df = train_df.copy()
scored_train_df['anomaly_score'] = anomaly_validator.model.score_samples(scaled_train_features)

anomaly_scores = scored_train_df['anomaly_score'].values
score_p90 = np.percentile(anomaly_scores, 90)
score_p50 = np.percentile(anomaly_scores, 50)
score_p15 = np.percentile(anomaly_scores, 15)
score_p05 = np.percentile(anomaly_scores, 5)

def sample_suite(frame, n_rows):
    if len(frame) == 0:
        return frame
    return frame.sample(n=min(n_rows, len(frame)), random_state=42)

clean_examples_df = pd.concat([
    sample_suite(scored_train_df[scored_train_df['anomaly_score'] >= score_p90], 100),
    sample_suite(scored_train_df[(scored_train_df['anomaly_score'] >= score_p50) & (scored_train_df['anomaly_score'] < score_p90)], 150),
], ignore_index=True)
clean_validation_examples = clean_examples_df[tracking_feature_columns].to_dict(orient='records')
print(f"[Sampling] Built {len(clean_validation_examples)} clean validation examples from high-score rows.")

invalid_examples_df = pd.concat([
    sample_suite(scored_train_df[(scored_train_df['anomaly_score'] >= score_p05) & (scored_train_df['anomaly_score'] < score_p15)], 100),
    sample_suite(scored_train_df[scored_train_df['anomaly_score'] < score_p05], 150),
], ignore_index=True)
invalid_validation_examples = invalid_examples_df[tracking_feature_columns].to_dict(orient='records')
print(f"[Sampling] Built {len(invalid_validation_examples)} anomaly validation examples from low-score rows.")

clean_training_df = scored_train_df[scored_train_df['anomaly_score'] >= np.percentile(anomaly_scores, 25)].copy()
print(f"[Sampling] Keeping {len(clean_training_df)} rows for NN training after filtering low-confidence points.")

LEAK-PROOF HIERARCHICAL SPLIT COMPLETE
   * Total Dataset Base:          10169 rows
   * Real Tournament Pool (85%):  8643 rows
     ├── NN Training Split:       6914 rows
     └── NN Holdout Validation:   1729 rows
   * Isolated Evaluation Test Set: 1526 rows (15% Untouchable)
[Split] The isolated test set is not used anywhere else in this cell.

 Initiating multi-tiered validation suite sampling...
[Sampling] Training an isolation forest on the internal training split.
[Sampling] Built 250 clean validation examples from high-score rows.
[Sampling] Built 250 anomaly validation examples from low-score rows.
[Sampling] Keeping 5185 rows for NN training after filtering low-confidence points.


### Train Neural Network

In [ ]:
pipeline_weights = optimize_pipeline_weights(train_df, clean_validation_examples, invalid_validation_examples, iterations=200)
print("[Training] Learned feature weights from the sampled clean and invalid suites.")

nn_validator = NNValidator(epochs=120, batch_size=16, lr=0.005)
nn_validator.train_with_weights(clean_training_df[tracking_feature_columns], pipeline_weights, quiet=False)
print("[Training] Trained the NN validator on the filtered training subset.")

print("\n" + "=" * 60)
print("             PRODUCTION PERFORMANCE METRICS (LEAK-PROOF)")
print("=" * 60)

print("\n--- PHASE 1: UNSEEN CLEAN HOLDOUT TESTING (Sensitivity) ---")
print("[Evaluation] Measuring how often clean holdout rows are approved.")
holdout_pass_count = sum(nn_validator.validate_row(row.to_dict()) for _, row in holdout_df.iterrows())
holdout_accuracy = (holdout_pass_count / len(holdout_df)) * 100
print(f"--> Clean Holdout Accuracy: {holdout_accuracy:.2f}% ({holdout_pass_count}/{len(holdout_df)} clean shots approved)")

print("\n--- PHASE 2: ANOMALY ARCHETYPE ATTACK TESTING (Specificity) ---")
print("[Evaluation] Measuring how often known bad examples are rejected.")
caught_anomalies = sum(not nn_validator.validate_row(example) for example in invalid_validation_examples)
anomaly_detection_rate = (caught_anomalies / len(invalid_validation_examples)) * 100
print(f"--> Anomaly Detection Rate: {anomaly_detection_rate:.2f}% ({caught_anomalies}/{len(invalid_validation_examples)} caught)")
print("=" * 60)

--- Meta-Optimizer Running (200 epochs) ---
  Meta-Epoch 001 -> Contrast Gap: 3.51x
  Meta-Epoch 050 -> Contrast Gap: 3.79x
  Meta-Epoch 100 -> Contrast Gap: 3.98x
  Meta-Epoch 150 -> Contrast Gap: 4.12x
  Meta-Epoch 200 -> Contrast Gap: 4.23x
[Training] Learned feature weights from the sampled clean and invalid suites.
[NN Production] Model ready. Separation threshold: 0.5266
[Training] Trained the NN validator on the filtered training subset.

             PRODUCTION PERFORMANCE METRICS (LEAK-PROOF)

--- PHASE 1: UNSEEN CLEAN HOLDOUT TESTING (Sensitivity) ---
[Evaluation] Measuring how often clean holdout rows are approved.
--> Clean Holdout Accuracy: 85.14% (1472/1729 clean shots approved)

--- PHASE 2: ANOMALY ARCHETYPE ATTACK TESTING (Specificity) ---
[Evaluation] Measuring how often known bad examples are rejected.
--> Anomaly Detection Rate: 83.60% (209/250 caught)


### Fit Distribution Validator

In [ ]:
print("[Init] Fitting the distribution validator on the 85% pipeline pool.")

distribution_validator = DistributionValidator(contamination=0.01)
distribution_validator.fit(train_pool_df, tracking_feature_columns)

### Instantiate components for synthetic data generation

In [ ]:
print("\n Initializing components...")

anchored_examples = get_anchored_examples_text(train_pool_df, samples_per_group=2)
print("[Init] Retrieved anchored examples for the challenger prompt builder.")

challenger = Challenger(
    llm_url=GENERATE_ENDPOINT,
    model_id=MODEL_ID,
    anchored_examples=anchored_examples,
)

retriever = Retriever(
    historical_data=train_pool_df,
    feature_columns=tracking_feature_columns,
)

synthetic_generator = SyntheticGenerator(
    llm_url=GENERATE_ENDPOINT,
    model_id=MODEL_ID,
    feature_columns=tracking_feature_columns,
)

judge = Judge(
    dist_validator=distribution_validator,
    nn_validator=nn_validator,
    llm_url=GENERATE_ENDPOINT,
    model_id=MODEL_ID,
    z_threshold=3.5,
)

main_orchestrator = MainOrchestrator(
    challenger=challenger,
    retriever=retriever,
    generator=synthetic_generator,
    judge=judge,
)

print("\nSystem initialization complete.")


 Initializing components...
[Init] Fitting the distribution validator on the 85% pipeline pool.
--> DistributionValidator successfully trained on 8643 rows.
[Init] Retrieved anchored examples for the challenger prompt builder.
[Retriever] Preparing vector indexing for 8643 data rows...
[Retriever Progress] Indexed 1000/8643 tracking rows.
[Retriever Progress] Indexed 2000/8643 tracking rows.
[Retriever Progress] Indexed 3000/8643 tracking rows.
[Retriever Progress] Indexed 4000/8643 tracking rows.
[Retriever Progress] Indexed 5000/8643 tracking rows.
[Retriever Progress] Indexed 6000/8643 tracking rows.
[Retriever Progress] Indexed 7000/8643 tracking rows.
[Retriever Progress] Indexed 8000/8643 tracking rows.
[Retriever Progress] Indexed 8643/8643 tracking rows.

System initialization complete.


### Generate synthetic data

In [ ]:
# generate synthetic data

TARGET_COL = 'Total'

# Safely pull feature columns directly from your master feature list
FEATURE_COLS = [c for c in feature_cols if c != TARGET_COL]

NUM_SAMPLES = 2

print("   -> Starting synthetic data generation")
agentic_rows = []
while len(agentic_rows) < NUM_SAMPLES:
    proposal = main_orchestrator.run_generation_pipeline(max_iterations=8)
    if proposal and proposal.get("status") == "SUCCESS":
        agentic_rows.append(proposal["synthetic_record"])

from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.impute import IterativeImputer

all_cols = FEATURE_COLS + [TARGET_COL]
imputer = IterativeImputer(max_iter=10, random_state=42)
imputer.fit(train_pool_df[all_cols].apply(pd.to_numeric, errors='coerce'))

df_only_agentic = pd.DataFrame(agentic_rows)
if TARGET_COL not in df_only_agentic.columns:
    print(f"⚠️ WARNING: '{TARGET_COL}' missing from Agentic output! Adding NaN column for imputation.")
    df_only_agentic[TARGET_COL] = float('nan')
for col in all_cols:
    df_only_agentic[col] = pd.to_numeric(df_only_agentic.get(col, float('nan')), errors='coerce')
df_only_agentic[all_cols] = imputer.transform(df_only_agentic[all_cols])
df_synth_agentic = pd.concat([train_pool_df, df_only_agentic], ignore_index=True)
print(f"Generated {len(df_only_agentic)} synthetic rows; combined training set has {len(df_synth_agentic)} rows.")
print(agentic_rows)

for row in agentic_rows:
    print(row)

   -> Starting synthetic data generation
INITIALIZING SELF-CORRECTING DATA ORCHESTRATION LOOP

[ITERATION 1/8]
----------------------------------------
[data_challenger] Compiling contextual prompt...
[data_challenger] Dispatching task to Ollama (qwen3.5:9b)...
[Orchestrator -> Challenger] Targeting Scenario:
 "Analyze a specific impact event where the golfer executes an aggressive low-to-high swing path designed to generate backspin while minimizing sidespin deviation from the intended target line (straight shot). The scenario requires identifying data points where the Club Path is steeply upward relative to neutral, specifically exceeding +3.0 degrees, combined with a Face Angle that remains closed relative to the face-to-path differential of less than 2.5 degrees absolute magnitude. Furthermore, constrain the analysis to instances where Impact Offset indicates an inside-out approach (positive value) greater than -7.64 inches but below zero, and ensure the Swing Plane is shallower th